# M4b — multi-label por técnica con XGBoost (Fase 2 SOTA)

**Idea (`docs/sota_attck_multilabel.md` §4):** el lineal M4 no separa T1041/T1567 (F1 0.10). Aquí va un **XGBoost binario por técnica** con el mismo X de M2 (TF-IDF 5k + tabular P1–P3), igual que M3 hace un modelo por tipo de nodo. `scale_pos_weight` por label para la cola; labels sin soporte en train → ceros + nota.

Mismo protocolo: threat live, 6 técnicas [T1190,T1041,T1567,T1204,T1486,T1566], split temporal 80/20, compara vs M4-LogReg (`models/m4_metrics.json`) y oráculo-tier.

In [1]:
import json
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd()
ROOT = ROOT if (ROOT / 'pyproject.toml').exists() else ROOT.parent

COLS = ['label_binary','timestamp','event_time','message_sanitized','attack_techniques',
        'message_type','stream_name','product_name','vendor_name','pipeline',
        'org_id','action','severity','protocol']
df = pd.read_parquet(ROOT / 'data' / 'signals' / 'signals.parquet', columns=COLS)
th = df[df.label_binary != 'benign'].copy().reset_index(drop=True)
TECHS = ['T1190','T1041','T1567','T1204','T1486','T1566']
def parse(v):
    try:
        t = json.loads(v) if v else []
        return [x for x in t if x in TECHS]
    except Exception:
        return []
th['tech_list'] = th.attack_techniques.fillna('').map(parse)
print('threat:', th.shape, '| tecnicas:', {t: int(sum(th.tech_list.map(lambda l: t in l))) for t in TECHS})

CATS = ['message_type','stream_name','product_name','vendor_name','pipeline',
        'org_id','action','severity','protocol']
th['hour'] = pd.to_datetime(th['timestamp'], unit='s', utc=True).dt.hour
th['msg_len'] = th.message_sanitized.fillna('').str.len()
th['lag_h'] = ((th['timestamp'] - th['event_time']) / 3600).clip(-1e4, 1e4).fillna(-1)
for c in CATS:
    th[c] = th[c].fillna('').astype(str)
msg = th.message_sanitized.fillna('')
cutoff = th['timestamp'].quantile(0.8)
tr = (th['timestamp'] <= cutoff).to_numpy(); te = ~tr
print(f'train/test: {tr.sum()}/{te.sum()}')

from sklearn.preprocessing import MultiLabelBinarizer
Y = MultiLabelBinarizer(classes=TECHS).fit_transform(th.tech_list)
print('train positives por label:', {t: int(Y[tr][:, j].sum()) for j, t in enumerate(TECHS)})

threat: (114521, 15) | tecnicas: {'T1190': 114507, 'T1041': 7742, 'T1567': 7728, 'T1204': 20, 'T1486': 0, 'T1566': 0}
train/test: 91617/22904


train positives por label: {'T1190': 91615, 'T1041': 4009, 'T1567': 4007, 'T1204': 10, 'T1486': 0, 'T1566': 0}


In [2]:
# X identico a M2: TF-IDF 5k (fit train) + tabular one-hot
from scipy.sparse import hstack
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder

vec = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), sublinear_tf=True, min_df=5)
Ttr = vec.fit_transform(msg[tr].tolist()); Tte = vec.transform(msg[te].tolist())
tab = ColumnTransformer([('cat', OneHotEncoder(handle_unknown='ignore', max_categories=30), CATS),
                           ('num', 'passthrough', ['hour','msg_len','lag_h'])])
Atr = tab.fit_transform(th[tr][CATS+['hour','msg_len','lag_h']])
Ate = tab.transform(th[te][CATS+['hour','msg_len','lag_h']])
Xtr = hstack([Ttr, Atr]).tocsr(); Xte = hstack([Tte, Ate]).tocsr()
print('X:', Xtr.shape, Xte.shape)

X: (91617, 5054) (22904, 5054)


In [3]:
# 6 XGBoost binarios, uno por tecnica (scale_pos_weight = neg/pos)
import time
from xgboost import XGBClassifier

t0 = time.time()
P = np.zeros_like(Y[te])
skipped = []
for j, t in enumerate(TECHS):
    pos = int(Y[tr][:, j].sum()); neg = int(tr.sum()) - pos
    if pos < 2:
        skipped.append(t)
        continue
    clf = XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.08,
                          subsample=0.8, colsample_bytree=0.8, tree_method='hist',
                          objective='binary:logistic', scale_pos_weight=neg / pos,
                          eval_metric='logloss', n_jobs=-1, random_state=42)
    clf.fit(Xtr, Y[tr][:, j])
    P[:, j] = (clf.predict_proba(Xte)[:, 1] >= 0.5).astype(int)
    print(f'{t}: pos train {pos}  spw {neg/pos:.0f}', flush=True)
print(f'skipped (sin soporte train): {skipped} -> ceros')
print(f'total: {time.time()-t0:.0f}s')

T1190: pos train 91615  spw 0


T1041: pos train 4009  spw 22


T1567: pos train 4007  spw 22


T1204: pos train 10  spw 9161


skipped (sin soporte train): ['T1486', 'T1566'] -> ceros
total: 158s


In [4]:
from sklearn.metrics import (classification_report, f1_score, hamming_loss,
                             accuracy_score, average_precision_score)
print('=== M4b XGBoost por tecnica ===')
print(classification_report(Y[te], P, target_names=TECHS, zero_division=0))
res = {'model': 'm4b-xgb-perlabel-v1', 'labels': TECHS,
       'micro_f1': round(float(f1_score(Y[te], P, average='micro')), 4),
       'macro_f1': round(float(f1_score(Y[te], P, average='macro')), 4),
       'hamming': round(float(hamming_loss(Y[te], P)), 4),
       'subset_acc': round(float(accuracy_score(Y[te], P)), 4),
       'n_train': int(tr.sum()), 'n_test': int(te.sum())}
print(res)
for j, t in enumerate(TECHS):
    try:
        ap = average_precision_score(Y[te][:, j], P[:, j])
    except Exception:
        ap = 0.0
    print(f'{t}: soporte {int(Y[te][:, j].sum())}  PR-AUC {round(float(ap), 4)}')
print('\nM4-LogReg:', json.loads((ROOT / 'models' / 'm4_metrics.json').read_text()))
(ROOT / 'models' / 'm4b_metrics.json').write_text(json.dumps(res, indent=2))
print('guardado models/m4b_metrics.json')

=== M4b XGBoost por tecnica ===
              precision    recall  f1-score   support

       T1190       1.00      1.00      1.00     22892
       T1041       0.08      0.06      0.07      3733
       T1567       0.08      0.06      0.07      3721
       T1204       0.24      1.00      0.39        10
       T1486       0.00      0.00      0.00         0
       T1566       0.00      0.00      0.00         0

   micro avg       0.82      0.77      0.79     30356
   macro avg       0.23      0.35      0.26     30356
weighted avg       0.77      0.77      0.77     30356
 samples avg       0.92      0.90      0.87     30356

{'model': 'm4b-xgb-perlabel-v1', 'labels': ['T1190', 'T1041', 'T1567', 'T1204', 'T1486', 'T1566'], 'micro_f1': 0.7926, 'macro_f1': 0.2557, 'hamming': 0.089, 'subset_acc': 0.7334, 'n_train': 91617, 'n_test': 22904}
T1190: soporte 22892  PR-AUC 0.9995
T1041: soporte 3733  PR-AUC 0.158
T1567: soporte 3721  PR-AUC 0.1574
T1204: soporte 10  PR-AUC 0.2439
T1486: soporte 0  P

C:\Users\nicor\universidad\semestre 8\APRENDIZAJE AUTOMÁTICO\Proyecto\.venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\nicor\universidad\semestre 8\APRENDIZAJE AUTOMÁTICO\Proyecto\.venv\lib\site-packages\sklearn\metrics\_ranking.py:1046: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
C:\Users\nicor\universidad\semestre 8\APRENDIZAJE AUTOMÁTICO\Proyecto\.venv\lib\site-packages\sklearn\metrics\_ranking.py:1046: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


## Lectura (medido)

- **T1190: 1.00/1.00/1.00.** Trivial, como M4.
- **T1041/T1567: P 0.08 / R 0.06 / F1 0.07 (peor que M4-LogReg 0.10).** El XGB conservador ni dispara: ni lineal ni no-lineal separan estas cabezas a nivel evento con estos features. Confirma la tesis: en live son priors de regla derivados del tier, no atribución aprendible por evento.
- **T1204: P 0.24 / R 1.00 / F1 0.39** (10/10 atrapados, mejor que M4). Cola detectable solo a base de FPs.
- **T1486/T1566: 0**, sin soporte en live (skipped en train).
- **Global: micro 0.79 > M4 0.75, macro 0.26 ≈ M4 0.25, hamming 0.089, subset-acc 0.73.** Mejora global por dominancia de T1190, misma macro por cola+FPs.
- **Veredicto: NO desplegable** como segunda cabeza (las P 0.08 mentirían al analista). Queda como experimento + paper. Siguiente real: sentence-level + MiniLM/SciBERT (Fase 2 resto / Fase 3 TRAM) y aumento sintético TTPXHunter con `incident_signals` para la cola; el mapeo tier→técnicas (oráculo 0.9994) sigue siendo la capa honesta en `/score`.